In [1]:
import polars as pl
import os
import datetime as dt
from pathlib import Path
import sys
cwd = Path.cwd()
while 'notebooks' in str(cwd):
	cwd = cwd.parent
sys.path.insert(1, str(cwd)) 

In [2]:
from src_strategy.configs.dataconfig import input_output_config_4

In [3]:
input_output_config_4.input_file_names.HEADERS

['EncounterEpicCsn',
 'Event_DateTime',
 'Event_ID',
 'Type',
 'Event_Grouper',
 'Event_Name',
 'NumericValue',
 'Value']

In [4]:
input_output_config_4.data_path

'/work/InternalMedicine/s223850/Sepsis-data/data/raw_data_phase4'

In [4]:
p3 = "../../../Sepsis-data/data/output/output_data_phase3_1/clinical_runs/phase3_2026-06-05/first_per_organ_type/20260921T200446Z__64827252f159__cfg-2c071cfa1e67"
p4 = "../../../Sepsis-data/data/output/output_data_phase4/clinical_runs/phase4_2026-06-05/first_per_organ_type/20261114T212724Z__70421eeb2096__cfg-2debc358c056"

In [5]:
df_sepsis_4 = pl.read_parquet(os.path.join(p4, 'df_sepsis_encounters.parquet'))

In [8]:
df_sepsis_4 = df_sepsis_4.filter(~pl.col("Sepsis_Category").str.starts_with("UPOA"))

In [9]:
df_enc_exl = pl.read_parquet(os.path.join(p4, 'df_encounter_exclusions.parquet'))
df_enc_exl

EncounterEpicCsn,exclusion_rule,source_column,operator,configured_value,observed_value
i64,str,str,str,str,str
753446467,"""discharge_after_2026_09_30""","""DischargeDateValue""","""gt""","""""2026-09-30""""","""2026-10-06"""
753858064,"""discharge_after_2026_09_30""","""DischargeDateValue""","""gt""","""""2026-09-30""""","""2026-10-01"""
755558318,"""discharge_after_2026_09_30""","""DischargeDateValue""","""gt""","""""2026-09-30""""","""2026-10-23"""
756322651,"""discharge_after_2026_09_30""","""DischargeDateValue""","""gt""","""""2026-09-30""""","""2026-10-03"""
757979394,"""discharge_after_2026_09_30""","""DischargeDateValue""","""gt""","""""2026-09-30""""","""2026-10-06"""
…,…,…,…,…,…
762871339,"""discharge_after_2026_09_30""","""DischargeDateValue""","""gt""","""""2026-09-30""""","""2026-10-21"""
762891543,"""discharge_after_2026_09_30""","""DischargeDateValue""","""gt""","""""2026-09-30""""","""2026-10-05"""
762894184,"""discharge_after_2026_09_30""","""DischargeDateValue""","""gt""","""""2026-09-30""""","""2026-10-01"""


In [13]:
df_s1 = pl.read_parquet(os.path.join(p3, "df_sepsis1_encounters.parquet"))
df_s2 = pl.read_parquet(os.path.join(p3, "df_sepsis2_encounters.parquet"))
df_s3 = pl.read_parquet(os.path.join(p3, "df_sepsis3_encounters.parquet"))
df_enc = pl.read_parquet(os.path.join("../../../Sepsis-data/data/output/output_data_phase3_1/", "df_encounters.parquet"))

In [10]:
df_s3_earliest = df_s3.filter(pl.col("sepsis_3_flag")==1).with_columns(
    pl.lit("3").alias("sepsis_score")
).select("EncounterEpicCsn", "sepsis_score", pl.col("sepsis_3_earliest_evidence_dt").alias("sepsis_dt"))
df_s2_earliest = df_s2.filter(pl.col("sepsis_2_flag")==1).with_columns(
    pl.lit("2").alias("sepsis_score")
).select("EncounterEpicCsn", "sepsis_score", pl.col("sepsis_2_earliest_evidence_dt").alias("sepsis_dt"))

df_s1_earliest = df_s1.filter(pl.col("sepsis_1_flag")==1).with_columns(
    pl.lit("1").alias("sepsis_score")
).select("EncounterEpicCsn", "sepsis_score", pl.col("sepsis_1_earliest_evidence_dt").alias("sepsis_dt"))

df_sepsis_p3 = pl.concat([df_s1_earliest, df_s2_earliest, df_s3_earliest])

In [ ]:
df_sepsis_p3_earliestonset_maxscore = df_sepsis_p3.group_by(
    "EncounterEpicCsn"
).agg(
    pl.col("sepsis_dt").min(),
    pl.col("sepsis_score").max()
)

In [25]:
df_sepsis_p3_earliestonset_maxscore = df_sepsis_p3_earliestonset_maxscore.join(
    df_enc,
    on="EncounterEpicCsn",
    how='inner'
)

In [32]:
df_sepsis_p3_earliestonset_maxscore = df_sepsis_p3_earliestonset_maxscore.with_columns(
    pl.when(pl.col("sepsis_dt") <= (pl.col("Arrival_Instant")+pl.duration(hours=48)))
    .then(pl.lit("POA-"))
    .otherwise(pl.lit("NPOA-"))
    .alias("POA-cat")
).with_columns(
    pl.concat_str([pl.col("POA-cat"), pl.col("sepsis_score")]).alias("sepsis_categorization")
)

In [40]:
df_sepsis_p3_earliestonset_maxscore.select(
    "EncounterEpicCsn",
	"sepsis_categorization"
).write_csv("phase3dot1_outcome.csv")

In [7]:
df_sepsis_4.select(
    "EncounterEpicCsn",
	pl.col("calculated_sepsis_category").alias("sepsis_categorization")
).write_csv("phase_4_outcome.csv")

In [42]:
df_sepsis_4['Arrival_Instant'].min(), df_sepsis_4['Arrival_Instant'].max()

(datetime.datetime(2022, 10, 1, 4, 54), datetime.datetime(2026, 9, 30, 18, 41))

In [44]:
df_sepsis_4.shape

(15133, 31)

In [46]:
df_enc_4 = pl.read_csv(os.path.join("../../../Sepsis-data/data/raw_data_phase4/Encounter Attributes - 10.28.26.csv"),
                       )

In [53]:
df_enc_4.columns

['PrimaryMrn',
 'PatientAgeAtAdmission',
 'Sex',
 'Ethnicity',
 'FirstRace',
 'MultiRacial',
 'EncounterEpicCsn',
 'AdmissionDateValue',
 'DischargeDateValue',
 'Arrival_Instant',
 'FirstAdmissionOrderInstant',
 'InpatientAdmissionInstant',
 'Admitted_from_ED',
 'PatientClass',
 'InpatientAdmissionPatientClass',
 'HospitalService',
 'LengthOfStayInDays',
 'AdmittingDepartment',
 'DischargeDepartment',
 'AdmissionType',
 'AdmissionSource',
 'AdmissionOrigin',
 'PrincipalProblem',
 'PrimaryCodedDiagnosis',
 'PrimaryCodedProcedureKey',
 'Death_Flag',
 'Original_POA_Condition',
 'Sepsis_Category',
 'Baseline_SBP',
 'Baseline_RespiratoryRate',
 'Baseline_PulseRate',
 'Baseline_Creatinine',
 'Baseline_Platelets',
 'Baseline_Bilirubin',
 'Baseline_eGFR',
 'Baseline_WBC',
 'First_Suspected_Infection_Time',
 'Suspected_Infection_Criteria_Met',
 'Cancer_Registry_YN',
 'HIV_Registry_YN',
 'Immunocrompromised_Registry_YN',
 'CKD_Dialysis_Registry_YN',
 'Solid_Organ_Transplant_Registry_YN',
 'Pregn

In [63]:
df_enc_4.with_columns(
    pl.when(pl.col("FirstRace").str.to_lowercase().str.contains("asian"))
	.then(pl.lit("Asian combined"))
    .otherwise(pl.col("FirstRace"))
    .alias("FirstRaceMerged")
)['FirstRaceMerged'].value_counts(sort=True).with_columns(
    (pl.col("count")/pl.col("count").sum()).alias("perc")
)

FirstRaceMerged,count,perc
str,u64,f64
"""White""",10078,0.665962
"""Black or African American""",3493,0.23082
"""Asian combined""",734,0.048503
"""Unavailable/Unknown""",423,0.027952
"""Some other race""",254,0.016785
"""American Indian or Alaska Nati…",65,0.004295
"""Declined""",61,0.004031
"""Native Hawaiian or Other Pacif…",25,0.001652


In [ ]:
df_enc_4['FirstRace'].value_counts().with_columns(
    pl.when(pl.col())
).with_columns(
    (pl.col("count")/pl.col("count").sum()).alias("perc")
).sort(by='count', descending=True)

FirstRace,count,perc
str,u64,f64
"""White""",10078,0.665962
"""Black or African American""",3493,0.23082
"""Unavailable/Unknown""",423,0.027952
"""Asian""",346,0.022864
"""Some other race""",254,0.016785
…,…,…
"""East Asian (China, North/South…",111,0.007335
"""Southeast Asian (Thailand, Vie…",96,0.006344
"""American Indian or Alaska Nati…",65,0.004295


### Why is CSN 761420581 NPOA and POA-3 

In [3]:
df_infect_4 = pl.read_parquet(os.path.join(p4, "df_infection_anchors.parquet"))

In [5]:
df_infect_4.filter(pl.col("EncounterEpicCsn")==761420581)

EncounterEpicCsn,infection_anchor_id,infection_episode_id,criterion,infect_dt,criterion_met_dt,suspicion_infection_type
i64,i64,str,str,datetime[μs],datetime[μs],str
761420581,1,"""FLOWSHEET_SUSPECTED_INFECTION:…","""first_ev_time_flowsheet""",2026-09-11 10:03:00,2026-09-11 10:03:00,"""FLOWSHEET_SUSPECTED_INFECTION"""
761420581,2,"""LACTATE+CULTURE:1""","""culture_dt+lactate_dt""",2026-09-11 10:06:29,2026-09-11 10:06:30,"""LACTATE+CULTURE"""
761420581,3,"""LACTATE+CULTURE:2""","""culture_dt+lactate_dt""",2026-09-11 10:06:29,2026-09-11 10:06:30,"""LACTATE+CULTURE"""
761420581,4,"""LACTATE+CULTURE:3""","""culture_dt+lactate_dt""",2026-09-11 10:06:29,2026-09-11 10:30:20,"""LACTATE+CULTURE"""
761420581,5,"""LACTATE+CULTURE:4""","""culture_dt+lactate_dt""",2026-09-23 21:24:34,2026-09-23 21:24:34,"""LACTATE+CULTURE"""
761420581,6,"""LACTATE+CULTURE:5""","""culture_dt+lactate_dt""",2026-09-23 21:24:34,2026-09-23 23:45:24,"""LACTATE+CULTURE"""


In [8]:
df_sepsis_4.filter(
    pl.col("EncounterEpicCsn") == 761420581
)

EncounterEpicCsn,Arrival_Instant,InpatientAdmissionInstant,LengthOfStayInDays,Death_Flag,Sepsis_Category,sepsis_1_flag,sepsis_1_earliest_evidence_dt,sepsis_1_dt,sepsis_1_association_count,sepsis_1_anchor_count,sepsis_1_sirs_episode_count,sepsis_2_flag,sepsis_2_earliest_evidence_dt,sepsis_2_dt,sepsis_2_association_count,sepsis_2_anchor_count,sepsis_2_organ_episode_count,sepsis_3_flag,sepsis_3_earliest_evidence_dt,sepsis_3_dt,sepsis_3_criterion_met_dt,sepsis_3_association_count,sepsis_3_anchor_count,sepsis_3_organ_episode_count,sepsis_3_shock_evidence_count,calculated_sepsis_severity,first_sepsis_dt,calculated_poa_status,calculated_sepsis_category,sepsis_category_match
i64,datetime[μs],datetime[μs],i64,i64,str,i8,datetime[μs],datetime[μs],i64,i64,i64,i8,datetime[μs],datetime[μs],i64,i64,i64,i8,datetime[μs],datetime[μs],datetime[μs],i64,i64,i64,i64,i8,datetime[μs],str,str,bool
761420581,2026-09-11 09:54:00,2026-09-12 14:20:00,25,0,"""POA-3""",1,2026-09-23 21:24:34,2026-09-23 21:24:34,12,2,6,0,null,null,0,0,0,0,null,null,null,0,0,0,0,1,2026-09-23 21:24:34,"""NPOA""","""NPOA-1""",false


In [52]:
df_vaso_4 = pl.read_parquet(os.path.join(p4, 'df_vasopressor_evidence.parquet'))
df_organdysf = pl.read_parquet(os.path.join(p4, 'df_organ_dysfunction.parquet'))

In [11]:
df_vaso_4.filter(
    pl.col("EncounterEpicCsn") == 761420581
)

EncounterEpicCsn,Event_DateTime,Event_ID,Type,Event_Grouper,Event_Name,NumericValue,vasopressor_order_time,vasopressor_administration_time,vasopressor_anchor_time,vasopressor_qualifying_dose_flag
i64,datetime[μs],i64,str,str,str,f64,datetime[μs],datetime[μs],datetime[μs],i8


In [18]:
from pathlib import Path
df_all_4 = pl.read_parquet(Path(p4).parent.parent.parent.parent/Path("df_all_no_collisions_map_measured_only.parquet"))

In [20]:
df_all_4.filter(
    (pl.col("EncounterEpicCsn") == 761420581)&
	(pl.col("Event_Grouper").is_in( ["Vasopressin",
            "Phenylephrine",
            "Norepinephrine",
            "Epinephrine",
            "Dopamine"]))
)

EncounterEpicCsn,Event_DateTime,Type,Event_ID,Event_Grouper,Event_Name,NumericValue,Value,sys,dia,map
i64,datetime[μs],str,i64,str,str,f64,str,f64,f64,f64


In [ ]:
df_all_4.filter(
    (pl.col("EncounterEpicCsn") == 761420581)
	&
    (pl.col("Event_DateTime") >= dt.datetime(2026, 9, 11, 9, 54))
)

EncounterEpicCsn,Event_DateTime,Type,Event_ID,Event_Grouper,Event_Name,NumericValue,Value,sys,dia,map
i64,datetime[μs],str,i64,str,str,f64,str,f64,f64,f64
761420581,2026-09-13 01:19:58,"""Procedure Order""",723918751,"""Blood Culture Order""","""CULTURE BLOOD""",null,null,null,null,null
761420581,2026-10-02 00:36:29,"""Procedure Order""",728182971,"""Lab""","""PROTHROMBIN TIME + INR""",null,null,null,null,null
761420581,2026-09-11 10:06:23,"""Procedure Order""",723694119,"""Imaging""","""CT ABDOMEN/PELVIS W IV CONTRAS…",null,null,null,null,null
761420581,2026-09-23 23:45:24,"""Procedure Order""",726403127,"""Lab""","""LACTATE (LACTIC ACID) SERUM""",null,null,null,null,null
761420581,2026-09-23 00:36:45,"""Procedure Order""",726115813,"""Lab""","""PROTHROMBIN TIME + INR""",null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…
761420581,2026-09-22 11:10:00,"""Flowsheet""",6679255967,"""Respirations""","""RESPIRATIONS""",16.0,"""16""",null,null,null
761420581,2026-09-21 14:20:00,"""Flowsheet""",6678456639,"""Respirations""","""RESPIRATIONS""",13.0,"""13""",null,null,null
761420581,2026-09-23 22:35:00,"""Flowsheet""",6683960649,"""Respirations""","""RESPIRATIONS""",20.0,"""20""",null,null,null


In [29]:
df_all_4.filter(
    (pl.col("EncounterEpicCsn") == 761420581)
)['Event_Grouper'].unique().to_list()

['Braden Scale',
 'Imaging',
 'Encounter Diagnosis',
 'Blood Culture Order',
 'Problem List',
 'Chest X-ray',
 'EKG',
 'Lab',
 'Other Body Fluid Culture Order',
 'Hospital Problem',
 'Blood Pressure',
 'Glasgow Coma Score',
 'O2 Delivery Room Air',
 'Temperature',
 'Sepsis Body Fluid Culture Order',
 'O2 Delivery Simple Face Mask',
 'Pulse',
 'O2 Delivery Nasal Cannula',
 'Admission',
 'Arterial Blood Pressure Mean',
 'Weight',
 'Respirations',
 'Microbiology',
 'O2 Flow Rate',
 'Suspected Infection',
 'Billing Diagnosis',
 'Billing Diagnosis - Confirmed Infection',
 'BMI',
 'Skin Assessment']

In [6]:
data_path = input_output_config_4.data_path

In [42]:
os.listdir(data_path)

['Lab Results - 10.8.26.csv',
 'Source of Infection Questions - 10.8.26.csv',
 'Diagnosis Events - 10.8.26.csv',
 'Encounter Attributes - 10.28.26.csv',
 'Flowsheet SIRS Vitals - 10.8.26.csv',
 'Encounter Table with Baseline Values - June 2022 - May 2026 - 6.5.26.csv',
 'Med Orders - 10.8.26.csv',
 'Procedure Orders - 10.8.26.csv',
 'Flowsheet Other - 10.8.26.csv',
 'Sepsis Events - 10.8.26.csv',
 'Med Admin - 10.8.26.csv']

In [ ]:
df_infect_4.filter(
    pl.col("EncounterEpicCsn") == 761420581
)

EncounterEpicCsn,infection_anchor_id,infection_episode_id,criterion,infect_dt,criterion_met_dt,suspicion_infection_type
i64,i64,str,str,datetime[μs],datetime[μs],str
761420581,1,"""FLOWSHEET_SUSPECTED_INFECTION:…","""first_ev_time_flowsheet""",2026-09-11 10:03:00,2026-09-11 10:03:00,"""FLOWSHEET_SUSPECTED_INFECTION"""
761420581,2,"""LACTATE+CULTURE:1""","""culture_dt+lactate_dt""",2026-09-11 10:06:29,2026-09-11 10:06:30,"""LACTATE+CULTURE"""
761420581,3,"""LACTATE+CULTURE:2""","""culture_dt+lactate_dt""",2026-09-11 10:06:29,2026-09-11 10:06:30,"""LACTATE+CULTURE"""
761420581,4,"""LACTATE+CULTURE:3""","""culture_dt+lactate_dt""",2026-09-11 10:06:29,2026-09-11 10:30:20,"""LACTATE+CULTURE"""
761420581,5,"""LACTATE+CULTURE:4""","""culture_dt+lactate_dt""",2026-09-23 21:24:34,2026-09-23 21:24:34,"""LACTATE+CULTURE"""
761420581,6,"""LACTATE+CULTURE:5""","""culture_dt+lactate_dt""",2026-09-23 21:24:34,2026-09-23 23:45:24,"""LACTATE+CULTURE"""


In [15]:
df_enc_4  = pl.read_csv(os.path.join(data_path, "Encounter Attributes - 10.28.26.csv"),
                         null_values=['null', "NULL", 'Null', 'NONE', 'None'],
                        infer_schema=True,
						# new_columns=input_output_config_4.input_file_names.HEADERS,
                        # has_header=False
                        )

df_proc_4  = pl.read_csv(os.path.join(data_path, "Procedure Orders - 10.8.26.csv"),
                         null_values=['null', "NULL", 'Null', 'NONE', 'None'],
                        infer_schema=True,
						new_columns=input_output_config_4.input_file_names.HEADERS,
                        has_header=False
                        )
df_meds_4 = pl.read_csv(os.path.join(data_path, "Med Admin - 10.8.26.csv"),
                         null_values=['null', "NULL", 'Null', 'NONE', 'None'],
                        infer_schema=False,
						new_columns=input_output_config_4.input_file_names.HEADERS,
                        has_header=False
                        )
df_medsord_4 = pl.read_csv(os.path.join(data_path, "Med Orders - 10.8.26.csv"),
                         null_values=['null', "NULL", 'Null', 'NONE', 'None'],
                        infer_schema=False,
						new_columns=input_output_config_4.input_file_names.HEADERS,
                        has_header=False
                        )
df_medsord_4 = pl.read_csv(os.path.join(data_path, "Med Orders - 10.8.26.csv"),
                         null_values=['null', "NULL", 'Null', 'NONE', 'None'],
                        infer_schema=False,
						new_columns=input_output_config_4.input_file_names.HEADERS,
                        has_header=False
                        )

In [16]:
df_medsord_4.filter(
    ((pl.col("EncounterEpicCsn").cast(pl.Int32)) == 671195009)
    &
    (pl.col("Event_Grouper").str.to_lowercase().str.contains("antib"))
).sort(by='Event_DateTime')

EncounterEpicCsn,Event_DateTime,Event_ID,Type,Event_Grouper,Event_Name,NumericValue,Value
str,str,str,str,str,str,str,str
"""671195009""","""2022-10-24 15:57:00.000""","""503426886""","""Medication Order""","""IV Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",null,null
"""671195009""","""2022-10-24 16:04:00.000""","""503428153""","""Medication Order""","""IV Antibiotics""","""PIPERACILLIN-TAZOBACTAM 4.5G I…",null,null
"""671195009""","""2022-10-24 16:04:00.000""","""503428154""","""Medication Order""","""IV Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",null,null
"""671195009""","""2022-10-24 16:32:00.000""","""503432870""","""Medication Order""","""IV Antibiotics""","""VANCOMYCIN IV 1500 MG (VIAL2BA…",null,null
"""671195009""","""2022-10-24 16:44:00.000""","""503434382""","""Medication Order""","""IV Antibiotics""","""PIPERACILLIN-TAZOBACTAM 4.5G I…",null,null
"""671195009""","""2022-10-24 17:26:00.000""","""503438903""","""Medication Order""","""IV Antibiotics""","""VANCOMYCIN 750 MG/150 ML IN DE…",null,null


In [18]:
df_proc_4.filter(
    ((pl.col("EncounterEpicCsn").cast(pl.Int32)) == 671195009)
    # &
    # (pl.col("Event_Grouper").str.to_lowercase().str.contains("antib"))
).sort(by='Event_DateTime')

EncounterEpicCsn,Event_DateTime,Event_ID,Type,Event_Grouper,Event_Name,NumericValue,Value
i64,str,i64,str,str,str,str,str
671195009,"""2022-10-22 21:52:48.000""",503252219,"""Procedure Order""","""EKG""","""EKG (12 LEAD)""",null,null
671195009,"""2022-10-22 21:52:53.000""",503252236,"""Procedure Order""","""Sepsis Body Fluid Culture Orde…","""UA + REFLEX TO MICROSCOPY""",null,null
671195009,"""2022-10-22 21:52:54.000""",503252241,"""Procedure Order""","""Lab""","""CBC W/ DIFF""",null,null
671195009,"""2022-10-22 21:52:54.000""",503252239,"""Procedure Order""","""EKG""","""EKG (12 LEAD)""",null,null
671195009,"""2022-10-22 21:52:54.000""",503252238,"""Procedure Order""","""Sepsis Body Fluid Culture Orde…","""UA + REFLEX TO MICROSCOPY""",null,null
…,…,…,…,…,…,…,…
671195009,"""2022-11-09 09:21:05.000""",505415063,"""Procedure Order""","""Chest X-ray""","""XR CHEST 1 VIEW""",null,null
671195009,"""2022-11-09 09:21:09.000""",505415079,"""Procedure Order""","""Chest X-ray""","""XR CHEST 1 VIEW""",null,null
671195009,"""2022-11-10 02:01:27.000""",505551686,"""Procedure Order""","""Lab""","""CBC W/ DIFF""",null,null


In [12]:
df_meds_4.filter(
    ((pl.col("EncounterEpicCsn").cast(pl.Int32)) == 671195009)
    &
    (pl.col("Event_Grouper").str.to_lowercase().str.contains("antib"))
).sort(by='Event_DateTime')

EncounterEpicCsn,Event_DateTime,Event_ID,Type,Event_Grouper,Event_Name,NumericValue,Value
str,str,str,str,str,str,str,str
"""671195009""","""2022-10-24 17:02:05.000""","""503428153""","""Medication Administration""","""IV Antibiotics - First""","""PIPERACILLIN-TAZOBACTAM 4.5G I…","""4.500""","""g"""
"""671195009""","""2022-10-24 17:32:07.000""","""503428153""","""Medication Administration""","""IV Antibiotics - Last""","""PIPERACILLIN-TAZOBACTAM 4.5G I…","""0.000""","""g"""
"""671195009""","""2022-10-24 17:41:31.000""","""503432870""","""Medication Administration""","""IV Antibiotics - First""","""VANCOMYCIN IV 1500 MG (VIAL2BA…","""1500.000""","""mg"""
"""671195009""","""2022-10-24 17:55:08.000""","""503432870""","""Medication Administration""","""IV Antibiotics""","""VANCOMYCIN IV 1500 MG (VIAL2BA…","""0.000""","""mg"""
"""671195009""","""2022-10-24 17:57:12.000""","""503432870""","""Medication Administration""","""IV Antibiotics""","""VANCOMYCIN IV 1500 MG (VIAL2BA…",null,"""*Unspecified"""
…,…,…,…,…,…,…,…
"""671195009""","""2022-10-27 10:00:00.000""","""503434382""","""Medication Administration""","""IV Antibiotics""","""PIPERACILLIN-TAZOBACTAM 4.5G I…",null,"""*Unspecified"""
"""671195009""","""2022-10-27 12:44:04.000""","""503438903""","""Medication Administration""","""IV Antibiotics - Last""","""VANCOMYCIN 750 MG/150 ML IN DE…",null,"""*Unspecified"""
"""671195009""","""2022-10-27 13:30:32.000""","""503434382""","""Medication Administration""","""IV Antibiotics""","""PIPERACILLIN-TAZOBACTAM 4.5G I…","""4.500""","""g"""


In [ ]:
df_enc_4.columns

['PrimaryMrn',
 'PatientAgeAtAdmission',
 'Sex',
 'Ethnicity',
 'FirstRace',
 'MultiRacial',
 'EncounterEpicCsn',
 'AdmissionDateValue',
 'DischargeDateValue',
 'Arrival_Instant',
 'FirstAdmissionOrderInstant',
 'InpatientAdmissionInstant',
 'Admitted_from_ED',
 'PatientClass',
 'InpatientAdmissionPatientClass',
 'HospitalService',
 'LengthOfStayInDays',
 'AdmittingDepartment',
 'DischargeDepartment',
 'AdmissionType',
 'AdmissionSource',
 'AdmissionOrigin',
 'PrincipalProblem',
 'PrimaryCodedDiagnosis',
 'PrimaryCodedProcedureKey',
 'Death_Flag',
 'Original_POA_Condition',
 'Sepsis_Category',
 'Baseline_SBP',
 'Baseline_RespiratoryRate',
 'Baseline_PulseRate',
 'Baseline_Creatinine',
 'Baseline_Platelets',
 'Baseline_Bilirubin',
 'Baseline_eGFR',
 'Baseline_WBC',
 'First_Suspected_Infection_Time',
 'Suspected_Infection_Criteria_Met',
 'Cancer_Registry_YN',
 'HIV_Registry_YN',
 'Immunocrompromised_Registry_YN',
 'CKD_Dialysis_Registry_YN',
 'Solid_Organ_Transplant_Registry_YN',
 'Pregn

In [70]:
df_sepsis_4 = df_sepsis_4.join(
	df_enc_4.select("EncounterEpicCsn", "ICU_YN").with_columns(
        pl.col("EncounterEpicCsn").cast(pl.Int32),
        pl.col("ICU_YN").cast(pl.Int32)
	), on='EncounterEpicCsn',
    how='left'
)

In [72]:
df_sepsis_4.columns

['EncounterEpicCsn',
 'Arrival_Instant',
 'InpatientAdmissionInstant',
 'LengthOfStayInDays',
 'Death_Flag',
 'Sepsis_Category',
 'sepsis_1_flag',
 'sepsis_1_earliest_evidence_dt',
 'sepsis_1_dt',
 'sepsis_1_association_count',
 'sepsis_1_anchor_count',
 'sepsis_1_sirs_episode_count',
 'sepsis_2_flag',
 'sepsis_2_earliest_evidence_dt',
 'sepsis_2_dt',
 'sepsis_2_association_count',
 'sepsis_2_anchor_count',
 'sepsis_2_organ_episode_count',
 'sepsis_3_flag',
 'sepsis_3_earliest_evidence_dt',
 'sepsis_3_dt',
 'sepsis_3_criterion_met_dt',
 'sepsis_3_association_count',
 'sepsis_3_anchor_count',
 'sepsis_3_organ_episode_count',
 'sepsis_3_shock_evidence_count',
 'calculated_sepsis_severity',
 'first_sepsis_dt',
 'calculated_poa_status',
 'calculated_sepsis_category',
 'sepsis_category_match',
 'ICU_YN']

In [ ]:
df_sepsis_4.group_by(
    "ICU_YN",
    "calculated_sepsis_category",
    "Sepsis_Category"
).agg(pl.len()).filter(~pl.col("Sepsis_Category").str.starts_with("UPOA")).filter(pl.col("ICU_YN")==1).pivot(on='Sepsis_Category', index='calculated_sepsis_category', values='len')

calculated_sepsis_category,POA-2,POA-1,NPOA-2,NPOA-3,POA-3,NPOA-1
str,u64,u64,u64,u64,u64,u64
"""NPOA-2""",2,8,10,13,3,33
"""POA-3""",268,380,56,732,2635,178
"""NPOA-3""",4,15,14,320,41,60
"""No Sepsis""",12,23,4,18,34,9
"""NPOA-1""",2,3,7,3,5,32
"""POA-1""",61,120,1,2,41,6
"""POA-2""",297,418,20,23,129,67


In [77]:
df_sepsis_4.group_by(
    "ICU_YN",
    "calculated_sepsis_category",
    "Sepsis_Category"
).agg(pl.len()).filter(~pl.col("Sepsis_Category").str.starts_with("UPOA")).filter(pl.col("ICU_YN")==0).pivot(on='Sepsis_Category', index='calculated_sepsis_category', values='len')

calculated_sepsis_category,NPOA-2,NPOA-3,POA-3,POA-2,NPOA-1,POA-1
str,u64,u64,u64,u64,u64,u64
"""POA-1""",7,2,21,544,63,2573
"""NPOA-2""",21,2,3,7,82,39
"""No Sepsis""",1,3,14,80,29,432
"""NPOA-1""",21,null,1,8,102,31
"""NPOA-3""",6,7,1,2,31,11
"""POA-3""",18,4,71,313,43,585
"""POA-2""",27,3,55,1039,148,2276


In [81]:
df_enc_4['Cancer_Registry_YN'].value_counts().with_columns(
    pl.col("count")/pl.col("count").sum()
)

Cancer_Registry_YN,count
str,f64
"""0""",0.640917
"""1""",0.359083


In [85]:
df_enc_4.with_columns(
    pl.col("DischargeDateValue").str.strptime(pl.Date, "%Y-%m-%d")
).filter(
    pl.col("DischargeDateValue")>dt.datetime(2026, 9, 30)
)

PrimaryMrn,PatientAgeAtAdmission,Sex,Ethnicity,FirstRace,MultiRacial,EncounterEpicCsn,AdmissionDateValue,DischargeDateValue,Arrival_Instant,FirstAdmissionOrderInstant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN,ICU_YN
str,str,str,str,str,str,str,str,date,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""71212527""","""53.626283""","""Male""","""Non-Hispanic/Latino""","""Black or African American""","""0""","""761520567""","""2026-09-23""",2026-10-07,"""2026-09-23 06:02:00.0000000""","""2026-09-23 11:06:45.000""","""2026-09-23 11:06:00.0000000""","""No""","""Inpatient""","""Inpatient""","""General Cardiology""","""14""","""UH 02O SICU""","""UH 09G""","""Elective""","""Home & Outside Location""","""OR Admission""","""Endocarditis of prosthetic mit…","""Thrombosis due to vascular pro…","""Performance of Urinary Filtrat…","""0""","""POA-1""","""POA-1""","""103""",null,"""83""","""13.2635714285714""","""77.9166666666667""","""0.675""","""4.57142857142857""","""6.3825""","""2026-09-24 15:00:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""1""","""0""","""0""","""1"""
"""93880679""","""73.727583""","""Female""","""Non-Hispanic/Latino""","""South Asian (India, Pakistan, …","""0""","""762899494""","""2026-09-30""",2026-10-08,"""2026-09-30 15:56:00.0000000""","""2026-09-30 20:28:06.000""","""2026-10-01 16:31:00.0000000""","""Yes""","""Inpatient""","""Observation""","""Hospital Medicine""","""8""","""UH 10G""","""UH 10G""","""Emergency""","""Home & Outside Location""","""ED Admission""","""Bacteremia""","""Sepsis, unspecified organism""","""Excision of Duodenum, Endo, Di…","""0""","""POA-1""","""POA-1""","""128""","""16""","""70""","""0.83625""","""262.625""","""0.38""","""74""","""12.69625""","""2026-09-30 17:52:19.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""1""","""0""","""1""","""0""","""0"""
"""90273611""","""65.530458""","""Female""","""Non-Hispanic/Latino""","""Black or African American""","""0""","""762086162""","""2026-09-21""",2026-10-05,"""2026-09-21 12:23:00.0000000""","""2026-09-21 17:29:10.000""","""2026-09-21 17:35:00.0000000""","""Yes""","""Inpatient""","""Inpatient""","""Hospital Medicine""","""14""","""UH 09G""","""UH 06O""","""Emergency""","""Home & Outside Location""","""ED Admission""","""*Unspecified""","""Leakage of other specified int…","""Wound Mgmt Trmt Musculosk Low …","""0""","""POA-1""","""POA-1""","""108""",null,null,"""0.805538461538462""","""316.818181818182""","""0.498333333333333""","""82.828125""","""11.4761538461538""","""2026-09-21 13:08:48.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""","""0"""
"""97770489""","""73.073237""","""Male""","""Non-Hispanic/Latino""","""East Asian (China, North/South…","""0""","""759135755""","""2026-08-12""",2026-10-03,"""2026-08-12 15:45:00.0000000""","""2026-08-12 15:53:20.000""","""2026-08-12 15:45:00.0000000""","""No""","""Inpatient""","""Inpatient""","""MICU""","""52""","""UH 12B""","""UH 08B MICU""","""Urgent""","""Short Term General Hospital""","""Transfer Center Admission""","""Shock""","""Unspecified cirrhosis of liver""","""Drainage of Peritoneal Cavity,…","""0""","""NPOA-3""","""NPOA-3""","""110""","""16""","""69""","""1.65389830508475""","""289.931818

In [69]:
df_enc_4['ICU_YN'].value_counts()

ICU_YN,count
str,u64
"""1""",6131
"""0""",9002


In [61]:
df_medsord_4.with_columns(
    pl.col("EncounterEpicCsn").cast(pl.Int32),
    pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")
).filter(
	pl.col("EncounterEpicCsn") == 761420581
)

/tmp/ipykernel_1268671/343050153.py:3: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")


EncounterEpicCsn,Event_DateTime,Event_ID,Type,Event_Grouper,Event_Name,NumericValue,Value
i32,datetime[μs],str,str,str,str,str,str


In [51]:
df_meds_4.with_columns(
    pl.col("EncounterEpicCsn").cast(pl.Int32),
    pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")
).filter(
	pl.col("EncounterEpicCsn") == 761420581
)

/tmp/ipykernel_1268671/3159009186.py:3: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")


EncounterEpicCsn,Event_DateTime,Event_ID,Type,Event_Grouper,Event_Name,NumericValue,Value
i32,datetime[μs],str,str,str,str,str,str


In [ ]:
df_organdysf.c

['EncounterEpicCsn',
 'Event_DateTime',
 'last_temp_8h',
 'last_temp_8h_source_ts',
 'last_pulse_8h',
 'last_pulse_8h_source_ts',
 'last_resp_8h',
 'last_resp_8h_source_ts',
 'last_sbp_8h',
 'last_sbp_8h_source_ts',
 'last_map_8h',
 'last_map_8h_source_ts',
 'last_wbc_12h',
 'last_wbc_12h_source_ts',
 'last_creatinine_12h',
 'last_creatinine_12h_source_ts',
 'last_creatinine_12h_source_type',
 'last_creatinine_12h_source_event_id',
 'last_creatinine_12h_source_event_name',
 'last_egfr_12h',
 'last_egfr_12h_source_ts',
 'last_egfr_12h_source_type',
 'last_egfr_12h_source_event_id',
 'last_egfr_12h_source_event_name',
 'last_bilirubin_12h',
 'last_bilirubin_12h_source_ts',
 'last_bilirubin_12h_source_type',
 'last_bilirubin_12h_source_event_id',
 'last_bilirubin_12h_source_event_name',
 'last_lactate_6h',
 'last_lactate_6h_source_ts',
 'last_lactate_6h_source_type',
 'last_lactate_6h_source_event_id',
 'last_lactate_6h_source_event_name',
 'last_platelets_24h',
 'last_platelets_24h_sourc

In [56]:
df_organdysf.filter(
    (pl.col("EncounterEpicCsn") == 761420581)
	&
    (pl.col("organ_dysfunction_total")==1)
).to_pandas()

,EncounterEpicCsn,Event_DateTime,last_temp_8h,last_temp_8h_source_ts,last_pulse_8h,last_pulse_8h_source_ts,last_resp_8h,last_resp_8h_source_ts,last_sbp_8h,last_sbp_8h_source_ts,...,bilirubin2x_criteria_flag,bilirubin_gt2_no_baseline_criteria_flag,hepatic_failure_flag,inr_criteria_flag,aptt_criteria_flag,platelets50_criteria_flag,platelets100_criteria_flag,coagulation_failure_flag,neurological_failure_flag,organ_dysfunction_total
0,761420581,2026-09-14 14:20:00,97.2,2026-09-14 14:20:00,74.0,2026-09-14 14:20:00,15.0,2026-09-14 14:20:00,141.0,2026-09-14 14:20:00,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,1,1
1,761420581,2026-09-14 14:22:00,97.2,2026-09-14 14:22:00,76.0,2026-09-14 14:22:00,18.0,2026-09-14 14:22:00,141.0,2026-09-14 14:22:00,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,1,1
2,761420581,2026-09-14 14:24:45,97.2,2026-09-14 14:22:00,76.0,2026-09-14 14:22:00,18.0,2026-09-14 14:22:00,141.0,2026-09-14 14:22:00,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,1,1
3,761420581,2026-09-14 14:25:00,97.2,2026-09-14 14:22:00,71.0,2026-09-14 14:25:00,16.0,2026-09-14 14:25:00,130.0,2026-09-14 14:25:00,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,1,1
4,761420581,2026-09-14 14:30:00,97.2,2026-09-14 14:22:00,70.0,2026-09-14 14:30:00,16.0,2026-09-14 14:30:00,133.0,2026-09-14 14:30:00,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,1,1
5,761420581,2026-09-14 14:35:00,97.2,2026-09-14 14:22:00,63.0,2026-09-14 14:35:00,15.0,2026-09-14 14:35:00,140.0,2026-09-14 14:35:00,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,1,1
6,761420581,2026-09-28 16:49:00,97.2,2026-09-28 16:49:00,68.0,2026-09-28 16:49:00,12.0,2026-09-28 16:49:00,98.0,2026-09-28 16:49:00,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,1,1
7,761420581,2026-09-28 16:51:00,97.2,2026-09-28 16:51:00,74.0,2026-09-28 16:51:00,22.0,2026-09-28 16:51:00,98.0,2026-09-28 16:51:00,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,1,1
8,761420581,2026-09-28 16:56:00,97.2,2026-09-28 16:51:00,79.0,2026-09-28 16:56:00,16.0,2026-09-28 16:56:00,79.0,2026-09-28 16:56:00,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,1,1
9,761420581,2026-09-28 17:01:00,97.2,2026-09-28 16:51:00,74.0,2026-09-28 17:01:00,20.0,2026-09-28 17:01:00,97.0,2026-09-28 17:01:00,...,NaN,NaN,0,NaN,NaN,NaN,NaN,0,1,1


In [59]:

df_organdysf.filter(
    (pl.col("EncounterEpicCsn") == 761420581)
	&
    (pl.col("last_lactate_6h").is_not_null())
).to_pandas()

,EncounterEpicCsn,Event_DateTime,last_temp_8h,last_temp_8h_source_ts,last_pulse_8h,last_pulse_8h_source_ts,last_resp_8h,last_resp_8h_source_ts,last_sbp_8h,last_sbp_8h_source_ts,...,bilirubin2x_criteria_flag,bilirubin_gt2_no_baseline_criteria_flag,hepatic_failure_flag,inr_criteria_flag,aptt_criteria_flag,platelets50_criteria_flag,platelets100_criteria_flag,coagulation_failure_flag,neurological_failure_flag,organ_dysfunction_total


In [14]:
df_sepsis_4 = df_sepsis_4.join(
    df_enc_4, on='EncounterEpicCsn', how='left', suffix='full_enc'
)

In [26]:
df_sepsis_4['Sex'].value_counts().with_columns(
	(pl.col("count")/pl.col("count").sum()).alias('prob')
)

Sex,count,prob
str,u64,f64
"""Male""",7838,0.531642
"""Female""",6905,0.468358


In [27]:
df_sepsis_4.with_columns(
    pl.when(pl.col("FirstRace").str.to_lowercase().str.contains("asian"))
	.then(pl.lit("Asian combined"))
    .otherwise(pl.col("FirstRace"))
    .alias("FirstRaceMerged")
)['FirstRaceMerged'].value_counts(sort=True).with_columns(
    (pl.col("count")/pl.col("count").sum()).alias("perc")
)

FirstRaceMerged,count,perc
str,u64,f64
"""White""",9825,0.666418
"""Black or African American""",3402,0.230754
"""Asian combined""",715,0.048498
"""Unavailable/Unknown""",407,0.027606
"""Some other race""",245,0.016618
"""American Indian or Alaska Nati…",65,0.004409
"""Declined""",59,0.004002
"""Native Hawaiian or Other Pacif…",25,0.001696


In [ ]:
# Red
34+10+5+47+9+6+26+6+42

185

In [29]:
212+222+508+3436+4493+5275

14146

In [30]:
842+207+1107+6448+2546+2996

14146

In [31]:
185/14146

0.013077901880390217

In [32]:
185/(6448+2546+2996)

0.015429524603836531

In [33]:
# Green
69+7+4+215+47+26+221+74+736

1399

In [35]:
1399/14146

0.09889721476035629

In [36]:
1399/(842+207+1107)

0.648886827458256

In [37]:
2.3+66.5

68.8